In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

REPO_ROOT = Path('/data/rbg/users/ashyamal/RNA-FM-benchmarking')
MODEL_OUT  = REPO_ROOT / 'model_outputs' / 'ICI'

FM_BULK = {'BulkFormer', 'BulkRNABert', 'COMPASS'}
FM_SC   = {'scGPT', 'STATE', 'CancerFoundation', 'UCE'}

WITHIN_COHORTS         = ['IMVigor210', 'IMMotion150']
ALL_COHORTS            = ['Hugo', 'Riaz', 'Rose', 'IMMotion150', 'IMVigor210']
TRANSFER_TRAIN_COHORTS = ['IMVigor210', 'IMMotion150']

COHORT_CANCER_TYPE = {
    'Hugo': 'SKCM', 'Riaz': 'SKCM',
    'Rose': 'BLCA', 'IMVigor210': 'BLCA',
    'IMMotion150': 'KIRC',
}
CANCER_TYPE_COHORTS = {}
for _c, _ct in COHORT_CANCER_TYPE.items():
    CANCER_TYPE_COHORTS.setdefault(_ct, []).append(_c)

SAME_CANCER_PAIRS = set()
for _cohorts in CANCER_TYPE_COHORTS.values():
    for _i, _a in enumerate(_cohorts):
        for _b in _cohorts[_i+1:]:
            SAME_CANCER_PAIRS.add((_a, _b))
            SAME_CANCER_PAIRS.add((_b, _a))

CATEGORY_ORDER = ['FM_BULK', 'FM_SC', 'GS', 'HVG', 'PCA', 'Raw_expression']
VERBOSE_ORDER  = ['BulkRNABert', 'BulkFormer', 'COMPASS',
                  'scGPT', 'STATE', 'CancerFoundation', 'UCE',
                  'GS', 'HVG', 'PCA', 'Raw_expression']

# Foundation models: blues (bulk RNA FMs) and purples (single-cell FMs)
# Baselines: warm oranges / muted neutrals — clearly distinct from FM family
PALETTE = {
    'FM_BULK':     '#1F78B4',
    'BulkRNABert': '#1F78B4',
    'BulkFormer':  '#1F78B4',
    'COMPASS':     '#1F78B4',
    'FM_SC':            '#74B9E7',
    'scGPT':            '#74B9E7',
    'STATE':            '#74B9E7',
    'CancerFoundation': '#74B9E7',
    'UCE':              '#74B9E7',
    'GS':             '#C4715A',
    'HVG':            '#C4715A',
    'PCA':            '#C4715A',
    'Raw_expression': '#C4715A',
}

# Sets used in the bar-chart legend
FM_VERBOSE       = {'BulkRNABert', 'BulkFormer', 'COMPASS', 'scGPT', 'STATE', 'CancerFoundation', 'UCE'}
BASELINE_VERBOSE = {'GS', 'HVG', 'PCA', 'Raw_expression'}

CANCER_TYPE_COLOR = {'SKCM': '#E91E63', 'BLCA': '#2196F3', 'KIRC': '#4CAF50'}

# Outputs (figures, per-task CSV) go to this repository; run outputs are read from REPO_ROOT
OUT_ROOT = Path('..').resolve()
FIG_DIR = OUT_ROOT / 'paper_figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
def categorize_row(rna_key, nHVGs):
    rk = str(rna_key)
    if rk in FM_BULK: return 'FM_BULK'
    if rk in FM_SC:   return 'FM_SC'
    if rk.startswith('max_') or rk.startswith('ssGSEA_'): return 'GS'
    if rk in ('PCA_50', 'PCA_100'): return 'PCA'
    if rk == 'RNA_seq':
        return 'Raw_expression' if str(nHVGs) in ('None', 'nan', '') else 'HVG'
    return 'Other'


RUN_KEY = ['cohort_tag', 'RNA_key', 'nHVGs', 'arch', 'seed', 'split']


def load_paradigm_metrics(paradigm):
    dfs = []
    base = MODEL_OUT / paradigm
    if not base.exists():
        return pd.DataFrame()
    for d in sorted(base.iterdir()):
        csv = d / 'all_metrics.csv'
        if csv.exists():
            dfs.append(pd.read_csv(csv))
    if not dfs:
        return pd.DataFrame()
    df = pd.concat(dfs, ignore_index=True)
    df['nHVGs'] = df['nHVGs'].fillna('None').astype(str)
    df = df.drop_duplicates(subset=RUN_KEY)  # earlier scoring passes appended some runs twice
    df['category'] = df.apply(lambda r: categorize_row(r['RNA_key'], r['nHVGs']), axis=1)
    return df


def load_val_metrics(paradigm):
    dfs = []
    base = MODEL_OUT / paradigm
    if not base.exists():
        return pd.DataFrame()
    for d in sorted(base.iterdir()):
        csv = d / 'val_metrics.csv'
        if csv.exists():
            dfs.append(pd.read_csv(csv))
        else:
            raise FileNotFoundError(f"Missing {csv}")
    df = pd.concat(dfs, ignore_index=True)
    df['nHVGs'] = df['nHVGs'].fillna('None').astype(str)
    df = df.drop_duplicates(subset=RUN_KEY)  # earlier scoring passes appended some runs twice
    return df

def select_best_config_globally(paradigm, valid_cohort_tags=None):
    val = load_val_metrics(paradigm)
    if val.empty or 'val_pr_auc' not in val.columns:
        return pd.DataFrame()
    if valid_cohort_tags is not None:
        val = val[val['cohort_tag'].isin(valid_cohort_tags)]
    val['nHVGs'] = val['nHVGs'].fillna('None').astype(str)
    val['category'] = val.apply(lambda r: categorize_row(r['RNA_key'], r['nHVGs']), axis=1)
    # FMs: select best arch per individual model; non-FMs: select best (nHVGs, arch) per category
    val['sel_group'] = val.apply(
        lambda r: r['RNA_key'] if r['category'] in ('FM_BULK', 'FM_SC') else r['category'], axis=1)
    val_mean = (
        val.dropna(subset=['val_pr_auc'])
        .groupby(['RNA_key', 'nHVGs', 'arch', 'sel_group'])['val_pr_auc']
        .mean().reset_index()
    )
    return (
        val_mean.sort_values('val_pr_auc', ascending=False)
        .groupby('sel_group', as_index=False).head(1)
        [['RNA_key', 'nHVGs', 'arch']]
    )

def aggregate_metrics(paradigm, valid_cohort_tags=None):
    df = load_paradigm_metrics(paradigm)
    if df.empty:
        print(f'{paradigm}: no metrics found')
        return pd.DataFrame()
    if valid_cohort_tags is not None:
        df = df[df['cohort_tag'].isin(valid_cohort_tags)]
    best = select_best_config_globally(paradigm, valid_cohort_tags=valid_cohort_tags)
    if not best.empty:
        df = df.merge(best, on=['RNA_key', 'nHVGs', 'arch'])  # global config
    group_cols = ['cohort_tag', 'RNA_key', 'nHVGs', 'arch', 'category']
    metric_cols = [c for c in df.columns if c.startswith('overall_')]
    agg_dict = {f'{c}_mean': (c, 'mean') for c in metric_cols}
    agg_dict.update({f'{c}_std': (c, 'std') for c in metric_cols})
    agg_dict['n_runs'] = ('overall_pr_auc', 'count')
    return (
        df.dropna(subset=['overall_pr_auc'])
        .groupby(group_cols).agg(**agg_dict).reset_index()
    )

_valid_within = [f'within_{c}' for c in WITHIN_COHORTS]
# Transfer is reported over the 8 source-target pairs with IMvigor210 or IMmotion150 as the source
# (Note S3, Table S10). Runs also exist for the other 12 pairs; they are excluded from selection and averages.
_valid_transfer = [f'transfer_{s}_to_{t}' for s in TRANSFER_TRAIN_COHORTS for t in ALL_COHORTS if t != s]

agg = {}
agg['within_cohort'] = aggregate_metrics('within_cohort', valid_cohort_tags=_valid_within)
agg['loco']          = aggregate_metrics('loco')
agg['transfer']      = aggregate_metrics('transfer', valid_cohort_tags=_valid_transfer)
# All 20 pairs, used only for the per-pair transfer heatmap
agg_transfer_all_pairs = aggregate_metrics('transfer')

for p, df in agg.items():
    print(f"{p}: {len(df)} rows" if not df.empty else f"{p}: no data yet")


within_cohort: 22 rows
loco: 55 rows
transfer: 88 rows


In [3]:
def select_best_arch_globally(paradigm, valid_cohort_tags=None):
    """Best arch per (RNA_key, nHVGs) averaged across cohorts — keeps all nHVGs."""
    val = load_val_metrics(paradigm)
    if val.empty or 'val_pr_auc' not in val.columns:
        return pd.DataFrame()
    if valid_cohort_tags is not None:
        val = val[val['cohort_tag'].isin(valid_cohort_tags)]
    val['nHVGs'] = val['nHVGs'].fillna('None').astype(str)
    val_mean = (
        val.dropna(subset=['val_pr_auc'])
        .groupby(['RNA_key', 'nHVGs', 'arch'])['val_pr_auc']
        .mean().reset_index()
    )
    return (
        val_mean.sort_values('val_pr_auc', ascending=False)
        .groupby(['RNA_key', 'nHVGs'], as_index=False).head(1)
        [['RNA_key', 'nHVGs', 'arch']]
    )

def aggregate_metrics_all_hvg(paradigm, valid_cohort_tags=None):
    """Like aggregate_metrics but keeps all nHVGs — for sensitivity plots only."""
    df = load_paradigm_metrics(paradigm)
    if df.empty:
        return pd.DataFrame()
    if valid_cohort_tags is not None:
        df = df[df['cohort_tag'].isin(valid_cohort_tags)]
    best = select_best_arch_globally(paradigm, valid_cohort_tags)
    if not best.empty:
        df = df.merge(best, on=['RNA_key', 'nHVGs', 'arch'])
    group_cols = ['cohort_tag', 'RNA_key', 'nHVGs', 'arch', 'category']
    metric_cols = [c for c in df.columns if c.startswith('overall_')]
    agg_dict = {f'{c}_mean': (c, 'mean') for c in metric_cols}
    agg_dict.update({f'{c}_std': (c, 'std') for c in metric_cols})
    agg_dict['n_runs'] = ('overall_pr_auc', 'count')
    return (
        df.dropna(subset=['overall_pr_auc'])
        .groupby(group_cols).agg(**agg_dict).reset_index()
    )

agg_hvg = {}
agg_hvg['within_cohort'] = aggregate_metrics_all_hvg('within_cohort', valid_cohort_tags=_valid_within)
agg_hvg['loco']          = aggregate_metrics_all_hvg('loco')
agg_hvg['transfer']      = aggregate_metrics_all_hvg('transfer', valid_cohort_tags=_valid_transfer)

In [4]:
# Random AUPRC baseline = positive fraction of the test population per cohort
import pickle

_ici_pkl = REPO_ROOT / 'data' / 'RNA_embeddings' / 'ICI' / 'ICI_all_processed.pkl'
with open(_ici_pkl, 'rb') as _f:
    _ici_data = pickle.load(_f)

pos_frac = {}
for cohort in ALL_COHORTS:
    responses = [v['response'] for v in _ici_data.values() if v.get('cohort') == cohort]
    pos_frac[cohort] = sum(responses) / len(responses)
    print(f"  {cohort}: pos_frac = {pos_frac[cohort]:.3f}  (n={len(responses)}, R={sum(responses)})")

# Paradigm-level random baseline = mean positive fraction over relevant test cohorts
# within_cohort: test cohorts are IMVigor210 + IMMotion150 only
# loco: each of the 5 cohorts is held out once
# transfer: targets of the 8 reported pairs (each training cohort -> the other 4 cohorts)
RANDOM_AUPRC = {
    'within_cohort': np.mean([pos_frac[c] for c in WITHIN_COHORTS]),
    'loco':          np.mean([pos_frac[c] for c in ALL_COHORTS]),
    'transfer':      np.mean([pos_frac[t] for s in TRANSFER_TRAIN_COHORTS for t in ALL_COHORTS if t != s]),
}
print("\nRandom AUPRC baselines (mean positive fraction):")
for p, v in RANDOM_AUPRC.items():
    print(f"  {p}: {v:.3f}")
del _ici_data  # free memory

  Hugo: pos_frac = 0.520  (n=25, R=13)
  Riaz: pos_frac = 0.196  (n=51, R=10)
  Rose: pos_frac = 0.180  (n=89, R=16)
  IMMotion150: pos_frac = 0.291  (n=165, R=48)
  IMVigor210: pos_frac = 0.228  (n=298, R=68)

Random AUPRC baselines (mean positive fraction):
  within_cohort: 0.260
  loco: 0.283
  transfer: 0.289


In [5]:
def best_per_category(df_agg, verbose=False):
    if df_agg.empty:
        return pd.DataFrame()
    df = df_agg.copy()
    if verbose:
        df['category'] = df.apply(
            lambda r: r['RNA_key'] if r['category'] in {'FM_BULK', 'FM_SC'} else r['category'], axis=1)
        order, allowed = VERBOSE_ORDER, set(VERBOSE_ORDER)
    else:
        order, allowed = CATEGORY_ORDER, set(CATEGORY_ORDER)
    df = df[df['category'].isin(allowed)]
    grouped = (
        df.groupby(['category', 'RNA_key', 'nHVGs', 'arch'])['overall_pr_auc_mean']
        .mean().reset_index().sort_values('overall_pr_auc_mean', ascending=False)
    )
    best = grouped.groupby('category', as_index=False).head(1)
    std  = df.groupby(['category', 'RNA_key', 'nHVGs', 'arch'])['overall_pr_auc_std'].mean().reset_index()
    best = best.merge(std, on=['category', 'RNA_key', 'nHVGs', 'arch'], how='left')
    best['category'] = pd.Categorical(best['category'], categories=order, ordered=True)
    return best.sort_values('category').reset_index(drop=True)


for p in ['within_cohort', 'loco', 'transfer']:
    if agg[p].empty: continue
    print(f"\n{'='*60}\n{p}\n{'='*60}")
    s = best_per_category(agg[p], verbose=True)
    print(s[['category','RNA_key','nHVGs','arch','overall_pr_auc_mean','overall_pr_auc_std']].to_string(index=False))



within_cohort
        category          RNA_key  nHVGs        arch  overall_pr_auc_mean  overall_pr_auc_std
     BulkRNABert      BulkRNABert   None ResidualMLP             0.410046            0.122855
      BulkFormer       BulkFormer   None ResidualMLP             0.392928            0.086641
         COMPASS          COMPASS   None      Linear             0.486213            0.123095
           scGPT            scGPT   None      Linear             0.387642            0.125070
           STATE            STATE   None ResidualMLP             0.441648            0.080084
CancerFoundation CancerFoundation   None ResidualMLP             0.442850            0.114406
             UCE              UCE   None ResidualMLP             0.457407            0.121864
              GS     max_combined   None ResidualMLP             0.484647            0.106258
             HVG          RNA_seq 2058.0 ResidualMLP             0.434421            0.070714
             PCA          PCA_100   None Resi

### Save results

In [6]:
# ── Per-task best AUPRC (one row per cohort_tag × category) ───────────────────
rows = []
for paradigm, df_agg in agg.items():
    if df_agg is None or df_agg.empty:
        continue
    for tag in sorted(df_agg['cohort_tag'].unique()):
        sub = df_agg[df_agg['cohort_tag'] == tag].copy()
        # best model config per category for this specific experiment
        idx  = sub.groupby('category')['overall_pr_auc_mean'].idxmax().dropna()
        best = sub.loc[idx].copy()
        best['paradigm'] = paradigm
        rows.append(best)

df_per_task = pd.concat(rows, ignore_index=True)

# parse cohort_tag into readable columns
df_per_task['held_out']    = df_per_task['cohort_tag'].str.extract(r'loco_held_out_(.+)')
df_per_task['train_cohort'] = df_per_task['cohort_tag'].str.extract(r'transfer_(.+)_to_.+')
df_per_task['test_cohort']  = df_per_task['cohort_tag'].str.extract(r'transfer_.+_to_(.+)')
df_per_task['within_cohort_id'] = df_per_task['cohort_tag'].str.extract(r'within_cohort_(.+)')

keep = ['paradigm', 'cohort_tag', 'within_cohort_id', 'held_out',
        'train_cohort', 'test_cohort', 'category', 'RNA_key', 'nHVGs',
        'arch', 'loss', 'overall_pr_auc_mean', 'overall_pr_auc_std']
keep = [c for c in keep if c in df_per_task.columns]
df_per_task = df_per_task[keep].sort_values(['paradigm', 'cohort_tag', 'category']).reset_index(drop=True)

out_path = OUT_ROOT / 'benchmarking_results' / 'ICI' / 'per_task_best_auprc.csv'
out_path.parent.mkdir(parents=True, exist_ok=True)
df_per_task.to_csv(out_path, index=False)
print(f"Saved {len(df_per_task)} rows → {out_path}")
display(df_per_task)

Saved 90 rows → /data/rbg/users/klingmin/projects/RNA-FM-benchmarking/benchmarking_results/ICI/per_task_best_auprc.csv


,paradigm,cohort_tag,within_cohort_id,held_out,train_cohort,test_cohort,category,RNA_key,nHVGs,arch,overall_pr_auc_mean,overall_pr_auc_std
0,loco,loco_held_out_Hugo,NaN,Hugo,NaN,NaN,FM_BULK,COMPASS,None,ResidualMLP,0.565826,0.021655
1,loco,loco_held_out_Hugo,NaN,Hugo,NaN,NaN,FM_SC,scGPT,None,ResidualMLP,0.716596,0.049906
2,loco,loco_held_out_Hugo,NaN,Hugo,NaN,NaN,GS,max_AURORA,None,ResidualMLP,0.540564,0.033900
3,loco,loco_held_out_Hugo,NaN,Hugo,NaN,NaN,HVG,RNA_seq,729.0,ResidualMLP,0.600120,0.042353
4,loco,loco_held_out_Hugo,NaN,Hugo,NaN,NaN,PCA,PCA_50,None,ResidualMLP,0.536791,0.021165
...,...,...,...,...,...,...,...,...,...,...,...,...
85,within_cohort,within_IMVigor210,NaN,NaN,NaN,NaN,FM_SC,CancerFoundation,None,ResidualMLP,0.535684,0.133131
86,within_cohort,within_IMVigor210,NaN,NaN,NaN,NaN,GS,max_combined,None,ResidualMLP,0.450039,0.131309
87,within_cohort,within_IMVigor210,NaN,NaN,NaN,NaN,HVG,RNA_seq,2058.0,ResidualMLP,0.453760,0.084797
88,within_cohort,within_IMVigor210,NaN,NaN,NaN,NaN,PCA,PCA_100,None,ResidualMLP,0.507458,0.091051


### Overview

In [7]:
import matplotlib.patches as mpatches

VALID_WITHIN_COHORTS = ['IMMotion150', 'IMVigor210']
VALID_WITHIN_TAGS    = [f'within_{c}' for c in VALID_WITHIN_COHORTS]

df_within_valid = agg['within_cohort'][
    agg['within_cohort']['cohort_tag'].isin(VALID_WITHIN_TAGS)
].copy()

print("Within-cohort (IMMotion150 + IMVigor210 only) — corrected best per category:")
print(best_per_category(df_within_valid, verbose=True)[
    ['category','RNA_key','nHVGs','arch','overall_pr_auc_mean','overall_pr_auc_std']
].to_string(index=False))

def plot_category_bar(paradigm, verbose=True, ax=None):
    df_agg = agg.get(paradigm)
    if df_agg is None or df_agg.empty:
        return
    summary = best_per_category(df_agg, verbose=verbose)
    if summary.empty:
        return
    cats   = summary['category'].tolist()
    means  = summary['overall_pr_auc_mean'].tolist()
    colors = [PALETTE.get(c, '#999999') for c in cats]
    if ax is None:
        fig, ax = plt.subplots(figsize=(max(6, len(cats) * 0.9), 4))
    x = np.arange(len(cats))
    bars = ax.bar(x, means, color=colors, edgecolor='white', linewidth=0.6, width=0.6)
    rand_val = RANDOM_AUPRC.get(paradigm, np.nan)
    if not np.isnan(rand_val):
        ax.axhline(rand_val, color='gray', linestyle='--', linewidth=1.2, zorder=5)
        rand_leg = ax.legend(
            handles=[Line2D([0], [0], color='gray', ls='--', lw=1.2,
                            label=f'Random AUPRC = {rand_val:.3f}')],
            loc='lower right', fontsize=8, framealpha=0.85)
        ax.add_artist(rand_leg)
    ax.set_xticks(x)
    ax.set_xticklabels(cats, rotation=35, ha='right', fontsize=9)
    ax.set_ylabel('AUPRC', fontsize=11)
    ax.set_ylim(0.25, 0.70)
    title = "LOCO" if paradigm == 'loco' else "Transfer" if paradigm == 'transfer' else paradigm.replace('_', ' ').title()
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(axis='y', ls=':', alpha=0.4)
    ax.spines[['top', 'right']].set_visible(False)
    for bar, m in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.008,
                f'{m:.3f}', ha='center', va='bottom', fontsize=7.5)

COHORT_SIZES = {'IMMotion150': 165, 'IMVigor210': 298}

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey='row')

# Row 1: exact same loop as before
for ax, cohort in zip(axes[0], VALID_WITHIN_COHORTS):
    tag = f'within_{cohort}'
    sub = df_within_valid[df_within_valid['cohort_tag'] == tag].copy()
    s   = best_per_category(sub, verbose=True)
    if s.empty:
        ax.set_title(cohort); continue
    cats   = s['category'].tolist()
    means  = s['overall_pr_auc_mean'].tolist()
    colors = [PALETTE.get(c, '#999999') for c in cats]
    x      = np.arange(len(cats))
    bars   = ax.bar(x, means, color=colors, edgecolor='white', linewidth=0.6, width=0.6)
    rand_val = pos_frac.get(cohort, np.nan)
    if not np.isnan(rand_val):
        ax.axhline(rand_val, color='gray', linestyle='--', linewidth=1.2, zorder=5)
        ax.legend(handles=[Line2D([0],[0], color='gray', ls='--', lw=1.2,
                                  label=f'Random AUPRC = {rand_val:.3f}')],
                  loc='lower right', fontsize=8, framealpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(cats, rotation=35, ha='right', fontsize=9)
    if ax is axes[0, 0]:
        ax.set_ylabel('AUPRC', fontsize=11)
    ax.set_ylim(0.2, 0.75)
    ax.set_title(f'Within-Cohort: {cohort} (n={COHORT_SIZES[cohort]})', fontsize=12, fontweight='bold')
    ax.grid(axis='y', ls=':', alpha=0.4)
    ax.spines[['top', 'right']].set_visible(False)
    for bar, m in zip(bars, means):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.008,
                f'{m:.3f}', ha='center', va='bottom', fontsize=7.5)

# Row 2: LOCO and Transfer
for ax, p in zip(axes[1], ['loco', 'transfer']):
    plot_category_bar(p, verbose=True, ax=ax)

axes[1, 1].set_ylabel('')

axes[1, -1].legend(handles=[
    mpatches.Patch(facecolor='#1F78B4', label='Bulk FMs'),
    mpatches.Patch(facecolor='#74B9E7', label='Single-cell FMs'),
    mpatches.Patch(facecolor='#C4715A', label='Baselines'),
], loc='upper right', fontsize=9, framealpha=0.85)

plt.tight_layout()
plt.savefig(FIG_DIR / 'ICI_auprc_overview.png', bbox_inches='tight', dpi=300)
plt.show()

Within-cohort (IMMotion150 + IMVigor210 only) — corrected best per category:
        category          RNA_key  nHVGs        arch  overall_pr_auc_mean  overall_pr_auc_std
     BulkRNABert      BulkRNABert   None ResidualMLP             0.410046            0.122855
      BulkFormer       BulkFormer   None ResidualMLP             0.392928            0.086641
         COMPASS          COMPASS   None      Linear             0.486213            0.123095
           scGPT            scGPT   None      Linear             0.387642            0.125070
           STATE            STATE   None ResidualMLP             0.441648            0.080084
CancerFoundation CancerFoundation   None ResidualMLP             0.442850            0.114406
             UCE              UCE   None ResidualMLP             0.457407            0.121864
              GS     max_combined   None ResidualMLP             0.484647            0.106258
             HVG          RNA_seq 2058.0 ResidualMLP             0.434421    

In [8]:
# FM vs baseline summary table
rows = []
for p in ['within_cohort', 'loco', 'transfer']:
    df_agg = agg.get(p)
    if df_agg is None or df_agg.empty: continue
    for _, row in best_per_category(df_agg, verbose=False).iterrows():
        rows.append({'paradigm': p, 'category': row['category'],
                     'pr_auc_mean': row['overall_pr_auc_mean'],
                     'pr_auc_std':  row['overall_pr_auc_std'],
                     'RNA_key': row['RNA_key'], 'nHVGs': row['nHVGs']})
summary_df = pd.DataFrame(rows)
if not summary_df.empty:
    summary_df['value'] = summary_df.apply(
        lambda r: f"{r['pr_auc_mean']:.3f} +/- {r['pr_auc_std']:.3f}"
                  if pd.notna(r['pr_auc_std']) else f"{r['pr_auc_mean']:.3f}", axis=1)
    display(summary_df.pivot(index='category', columns='paradigm', values='value').reindex(CATEGORY_ORDER))


paradigm,loco,transfer,within_cohort
category,,,
FM_BULK,0.382 +/- 0.046,0.393 +/- 0.041,0.486 +/- 0.123
FM_SC,0.377 +/- 0.031,0.373 +/- 0.038,0.457 +/- 0.122
GS,0.369 +/- 0.031,0.392 +/- 0.041,0.485 +/- 0.106
HVG,0.350 +/- 0.025,0.372 +/- 0.047,0.434 +/- 0.071
PCA,0.330 +/- 0.034,0.366 +/- 0.037,0.461 +/- 0.116
Raw_expression,0.359 +/- 0.038,0.375 +/- 0.049,0.462 +/- 0.079


### Within

In [9]:
# ── Within-cohort: restrict to intended cohorts (IMMotion150, IMVigor210) ──────
# 
# Root cause of inflated HVG score (0.916):
#   run_eval_ICI.sh evaluates ALL cohorts for within_cohort, including Hugo/Riaz/Rose
#   which have *stale* model outputs from a prior run. Hugo (n=25) achieves near-perfect
#   AUPRC with 44 HVGs. select_best_arch picks Linear for Hugo at nHVGs=44, but
#   ResidualMLP for IMMotion150/IMVigor210. best_per_category then averages (RNA_seq,
#   44, Linear) across only Hugo — reporting 0.916 instead of the true ~0.35.
#
# Fix: filter to the two large, intended within-cohort datasets only.

VALID_WITHIN_COHORTS = ['IMMotion150', 'IMVigor210']
VALID_WITHIN_TAGS    = [f'within_{c}' for c in VALID_WITHIN_COHORTS]

df_within_valid = agg['within_cohort'][
    agg['within_cohort']['cohort_tag'].isin(VALID_WITHIN_TAGS)
].copy()

print("Within-cohort (IMMotion150 + IMVigor210 only) — corrected best per category:")
print(best_per_category(df_within_valid, verbose=True)[
    ['category','RNA_key','nHVGs','arch','overall_pr_auc_mean','overall_pr_auc_std']
].to_string(index=False))

# ── Per-cohort bar charts (shared y-axis for comparability) ──────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)  # sharey keeps scale identical
for ax, cohort in zip(axes, VALID_WITHIN_COHORTS):
    size = 165 if cohort == 'IMMotion150' else 298
    tag = f'within_{cohort}'
    sub = df_within_valid[df_within_valid['cohort_tag'] == tag].copy()
    s   = best_per_category(sub, verbose=True)
    if s.empty:
        ax.set_title(cohort); continue
    cats   = s['category'].tolist()
    means  = s['overall_pr_auc_mean'].tolist()
    colors = [PALETTE.get(c, '#999999') for c in cats]
    x      = np.arange(len(cats))
    bars   = ax.bar(x, means, color=colors, edgecolor='white', linewidth=0.6, width=0.6)

    rand_val = pos_frac.get(cohort, np.nan)
    if not np.isnan(rand_val):
        ax.axhline(rand_val, color='gray', linestyle='--', linewidth=1.2, zorder=5)
        ax.legend(handles=[Line2D([0],[0], color='gray', ls='--', lw=1.2,
                                  label=f'Random AUPRC = {rand_val:.3f}')],
                  loc='lower right', fontsize=8, framealpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(cats, rotation=35, ha='right', fontsize=9)
    if ax is axes[0]:
        ax.set_ylabel('AUPRC', fontsize=11)
    ax.set_ylim(0.2, 0.75)
    ax.set_title(f'Within-Cohort: {cohort} (n={size})', fontsize=12, fontweight='bold')
    ax.grid(axis='y', ls=':', alpha=0.4)
    ax.spines[['top', 'right']].set_visible(False)
    for bar, m in zip(bars, means):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.008,
                f'{m:.3f}', ha='center', va='bottom', fontsize=7.5)

plt.tight_layout()
plt.savefig(FIG_DIR / 'ICI_within_per_cohort.png', bbox_inches='tight', dpi=300)
plt.show()

Within-cohort (IMMotion150 + IMVigor210 only) — corrected best per category:
        category          RNA_key  nHVGs        arch  overall_pr_auc_mean  overall_pr_auc_std
     BulkRNABert      BulkRNABert   None ResidualMLP             0.410046            0.122855
      BulkFormer       BulkFormer   None ResidualMLP             0.392928            0.086641
         COMPASS          COMPASS   None      Linear             0.486213            0.123095
           scGPT            scGPT   None      Linear             0.387642            0.125070
           STATE            STATE   None ResidualMLP             0.441648            0.080084
CancerFoundation CancerFoundation   None ResidualMLP             0.442850            0.114406
             UCE              UCE   None ResidualMLP             0.457407            0.121864
              GS     max_combined   None ResidualMLP             0.484647            0.106258
             HVG          RNA_seq 2058.0 ResidualMLP             0.434421    

### LOCO

In [10]:
# LOCO: per held-out cohort bar charts
df_loco = agg.get('loco', pd.DataFrame())
if df_loco.empty:
    print('No LOCO data yet')
else:
    tags = sorted(df_loco['cohort_tag'].unique())
    fig, axes = plt.subplots(2, 3, figsize=(15, 10), sharey=True)
    axes = axes.flatten()
    if len(tags) == 1: axes = [axes]
    for ax, tag in zip(axes, tags):
        sub = df_loco[df_loco['cohort_tag'] == tag]
        s   = best_per_category(sub, verbose=True)
        if s.empty: continue
        cats   = s['category'].tolist()
        means  = s['overall_pr_auc_mean'].tolist()
        colors = [PALETTE.get(c, '#999999') for c in cats]
        x    = np.arange(len(cats))
        bars = ax.bar(x, means, color=colors, edgecolor='white', linewidth=0.5)

        held_out = tag.replace('loco_held_out_', '')
        rand_val = pos_frac.get(held_out, np.nan)
        if not np.isnan(rand_val):
            ax.axhline(rand_val, color='gray', linestyle='--', linewidth=1.2, zorder=5)
            rand_handle = Line2D([0], [0], color='gray', linestyle='--', linewidth=1.2,
                                 label=f'Random AUPRC = {rand_val:.3f}')
            ax.legend(handles=[rand_handle], loc='lower right', fontsize=7, framealpha=0.8)

        ax.set_xticks(x)
        ax.set_xticklabels(cats, rotation=40, ha='right', fontsize=8)
        ax.set_title(held_out, fontsize=10, fontweight='bold')
        ax.set_ylabel('AUPRC')
        ax.set_ylim(0.2, 0.8)
        ax.grid(axis='y', alpha=0.3)
        for bar, m in zip(bars, means):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{m:.3f}',
                    ha='center', va='bottom', fontsize=6)
    #plt.suptitle('LOCO — AUPRC by Held-out Cohort', fontsize=13, y=1.02)
    axes[-1].set_visible(False)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'ICI_loco_per_cohort.png', bbox_inches='tight', dpi=150)
    plt.show()

In [11]:
# LOCO per-cohort table
import pandas as pd
df_best = pd.read_csv(OUT_ROOT / 'benchmarking_results' / 'ICI' / 'per_task_best_auprc.csv')
loco_best = df_best[df_best['paradigm'] == 'loco'].copy()
loco_best['cohort'] = loco_best['cohort_tag'].str.replace('loco_held_out_', '', regex=False)
pivot = loco_best.pivot(index='category', columns='cohort', values='overall_pr_auc_mean')
pivot = pivot.reindex(CATEGORY_ORDER)
pivot['mean'] = pivot.mean(axis=1)
display(pivot.round(3).sort_values('mean', ascending=False))

cohort,Hugo,IMMotion150,IMVigor210,Riaz,Rose,mean
category,,,,,,
FM_SC,0.717,0.355,0.309,0.276,0.361,0.403
FM_BULK,0.566,0.403,0.325,0.330,0.287,0.382
GS,0.541,0.413,0.328,0.284,0.279,0.369
Raw_expression,0.569,0.349,0.303,0.264,0.310,0.359
HVG,0.600,0.341,0.304,0.236,0.269,0.350
PCA,0.537,0.318,0.268,0.226,0.299,0.330


In [12]:
# LOCO stratified by cancer type of held-out cohort
df_loco = agg['loco'].copy()
df_loco['held_out']    = df_loco['cohort_tag'].str.replace('loco_held_out_', '', regex=False)
df_loco['cancer_type'] = df_loco['held_out'].map(COHORT_CANCER_TYPE)

ct_summary = (
    df_loco.groupby(['cancer_type', 'category'])['overall_pr_auc_mean']
    .mean().reset_index()
    .pivot(index='category', columns='cancer_type', values='overall_pr_auc_mean')
    .reindex(CATEGORY_ORDER)
)
ct_summary['mean'] = ct_summary.mean(axis=1)
print("AUPRC per category by cancer type of held-out cohort:")
display(ct_summary.round(3).sort_values('mean', ascending=False))
for ct, cohorts in CANCER_TYPE_COHORTS.items():
    print(f"  {ct}: {cohorts}")

# Random AUPRC per cancer type = mean positive fraction of cohorts in that cancer type
cancer_type_rand = {
    ct: np.mean([pos_frac[c] for c in cohorts if c in pos_frac])
    for ct, cohorts in CANCER_TYPE_COHORTS.items()
}

cancer_types = ['SKCM', 'BLCA', 'KIRC']
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
for ax, ct in zip(axes, cancer_types):
    cohorts_ct = CANCER_TYPE_COHORTS.get(ct, [])
    tags = [f'loco_held_out_{c}' for c in cohorts_ct]
    sub  = df_loco[df_loco['cohort_tag'].isin(tags)]
    if sub.empty:
        ax.set_title(ct); continue
    best_sub = (
        sub.groupby('category')['overall_pr_auc_mean']
        .agg(mean='mean', std='std').reset_index()
    )
    best_sub['category'] = pd.Categorical(best_sub['category'], categories=CATEGORY_ORDER, ordered=True)
    best_sub = best_sub.sort_values('category').dropna(subset=['mean'])
    cats   = best_sub['category'].tolist()
    means  = best_sub['mean'].tolist()
    colors = [PALETTE.get(c, '#999999') for c in cats]
    x    = np.arange(len(cats))
    bars = ax.bar(x, means, color=colors, edgecolor='white', linewidth=0.5)

    rand_val = cancer_type_rand.get(ct, np.nan)
    if not np.isnan(rand_val):
        ax.axhline(rand_val, color='gray', linestyle='--', linewidth=1.2, zorder=5)
        rand_handle = Line2D([0], [0], color='gray', linestyle='--', linewidth=1.2,
                             label=f'Random AUPRC = {rand_val:.3f}')
        ax.legend(handles=[rand_handle], loc='lower right', fontsize=8, framealpha=0.8)

    ax.set_xticks(x)
    ax.set_xticklabels(cats, rotation=40, ha='right', fontsize=9)
    ax.set_title(f'{ct}\n({", ".join(cohorts_ct)})', fontsize=10, fontweight='bold')
    ax.set_ylabel('AUPRC')
    ax.set_ylim(0.2, 0.8)
    ax.grid(axis='y', alpha=0.3)
    for bar, m in zip(bars, means):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{m:.3f}',
                ha='center', va='bottom', fontsize=7)
#plt.suptitle('LOCO — AUPRC Stratified by Cancer Type of Held-out Cohort', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'ICI_loco_by_cancer_type.png', bbox_inches='tight', dpi=150)
plt.show()

AUPRC per category by cancer type of held-out cohort:


cancer_type,BLCA,KIRC,SKCM,mean
category,,,,
GS,0.303,0.413,0.412,0.376
Raw_expression,0.307,0.349,0.416,0.357
HVG,0.286,0.341,0.418,0.349
FM_SC,0.275,0.321,0.433,0.343
FM_BULK,0.266,0.341,0.402,0.336
PCA,0.284,0.318,0.381,0.328


  SKCM: ['Hugo', 'Riaz']
  BLCA: ['Rose', 'IMVigor210']
  KIRC: ['IMMotion150']


### Transfer

In [13]:
# Transfer: per-pair best model AUPRC + same-cancer vs cross-cancer comparison
df_transfer = agg_transfer_all_pairs
if df_transfer.empty:
    print('No transfer data yet')
else:
    # For each (train→test) pair, take the single best model across all categories/configs
    best_per_pair = (
        df_transfer
        .sort_values('overall_pr_auc_mean', ascending=False)
        .groupby('cohort_tag', as_index=False).head(1)
        [['cohort_tag', 'overall_pr_auc_mean', 'category', 'RNA_key', 'nHVGs', 'arch']]
        .copy()
    )
    best_per_pair['train'] = (best_per_pair['cohort_tag']
                              .str.replace('transfer_', '', regex=False)
                              .str.split('_to_').str[0])
    best_per_pair['test']  = (best_per_pair['cohort_tag']
                              .str.replace('transfer_', '', regex=False)
                              .str.split('_to_').str[1])
    best_per_pair['same_cancer'] = best_per_pair.apply(
        lambda r: (r['train'], r['test']) in SAME_CANCER_PAIRS, axis=1)
    best_per_pair['type'] = best_per_pair['same_cancer'].map(
        {True: 'Same cancer type', False: 'Cross cancer type'})

    pivot     = best_per_pair.pivot(index='train', columns='test', values='overall_pr_auc_mean')
    pivot_cat = best_per_pair.pivot(index='train', columns='test', values='category')

    fig, ax = plt.subplots(1, 1, figsize=(8, 4))

    # ── Heatmap of per-pair best AUPRC ──────────────────────────────────
    im = ax.imshow(pivot.values, vmin=0.3, vmax=0.65, cmap='RdYlGn', aspect='auto')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_yticks(range(len(pivot.index)))
    ax.set_xticklabels(pivot.columns, rotation=30, ha='right')
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel('Test cohort')
    ax.set_ylabel('Train cohort')
    ax.set_title('Transfer — Best AUPRC per pair  (* = same cancer type)',
                 fontsize=10, fontweight='bold')
    plt.colorbar(im, ax=ax, label='AUPRC')

    for i, train in enumerate(pivot.index):
        for j, test in enumerate(pivot.columns):
            val = pivot.values[i, j]
            cat = pivot_cat.values[i, j]
            is_sc = (train, test) in SAME_CANCER_PAIRS
            if not np.isnan(val):
                ax.text(j, i - 0.15, f'{val:.2f}{"*" if is_sc else ""}',
                        ha='center', va='center', fontsize=9,
                        fontweight='bold' if is_sc else 'normal')
                if not pd.isna(cat):
                    ax.text(j, i + 0.22, str(cat),
                            ha='center', va='center', fontsize=6.5,
                            color='#222222', style='italic')
            if is_sc:
                ax.add_patch(plt.Rectangle((j-0.5, i-0.5), 1, 1,
                             fill=False, edgecolor='white', linewidth=2.5))

    plt.tight_layout()
    plt.savefig(FIG_DIR / 'ICI_transfer_heatmap.png', bbox_inches='tight', dpi=300)
    plt.show()

    print("\nSame-cancer vs cross-cancer (per-pair best model):")
    display(best_per_pair.groupby('type')['overall_pr_auc_mean']
            .agg(['mean','std','count']).round(3))
    print("\nSame-cancer pairs:")
    display(best_per_pair[best_per_pair['same_cancer']]
            [['train','test','category','RNA_key','overall_pr_auc_mean']].round(3))


Same-cancer vs cross-cancer (per-pair best model):


,mean,std,count
type,,,
Cross cancer type,0.403,0.154,16
Same cancer type,0.394,0.146,4



Same-cancer pairs:


,train,test,category,RNA_key,overall_pr_auc_mean
117,Riaz,Hugo,FM_SC,scGPT,0.613
17,Hugo,Riaz,FM_SC,scGPT,0.339
104,IMVigor210,Rose,FM_SC,CancerFoundation,0.315
153,Rose,IMVigor210,FM_SC,scGPT,0.310


In [14]:
# HVG sensitivity
PARADIGM_COLORS = {'within_cohort':'#E91E63','loco':'#2196F3','transfer':'#FF9800'}

hvg_rows = []
for p in ['within_cohort', 'loco', 'transfer']:
    df_agg = agg_hvg.get(p, pd.DataFrame())
    if df_agg.empty: continue
    hvg_df = df_agg[(df_agg['RNA_key']=='RNA_seq') & (df_agg['nHVGs']!='None') & df_agg['nHVGs'].notna()].copy()
    if hvg_df.empty: continue
    hvg_df['nHVGs_int'] = hvg_df['nHVGs'].astype(float).astype(int)
    mean_df = hvg_df.groupby('nHVGs_int')['overall_pr_auc_mean'].agg(pr_auc='mean', pr_auc_std='std').reset_index()
    mean_df['paradigm'] = p
    hvg_rows.append(mean_df)

raw_rows = []
for p in ['within_cohort', 'loco', 'transfer']:
    df_agg = agg.get(p, pd.DataFrame())
    if df_agg.empty: continue
    raw_df = df_agg[(df_agg['RNA_key']=='RNA_seq') & (df_agg['nHVGs']=='None')].copy()
    if raw_df.empty: continue
    raw_rows.append({'paradigm':p,'nHVGs_int':17490,'pr_auc':raw_df['overall_pr_auc_mean'].mean(),'pr_auc_std':raw_df['overall_pr_auc_mean'].std()})

hvg_summary = pd.concat(hvg_rows + ([pd.DataFrame(raw_rows)] if raw_rows else []), ignore_index=True)

if hvg_summary.empty:
    print('No HVG data yet')
else:
    fig, ax = plt.subplots(figsize=(9, 5))
    rand_handles = []
    for p in ['within_cohort', 'loco', 'transfer']:
        sub   = hvg_summary[hvg_summary['paradigm']==p].sort_values('nHVGs_int')
        color = PARADIGM_COLORS[p]
        ax.plot(sub['nHVGs_int'], sub['pr_auc'], marker='o', label=p.replace('_',' ').title(),
                color=color, linewidth=2, markersize=6)
        ax.fill_between(sub['nHVGs_int'],
                        sub['pr_auc']-sub['pr_auc_std'].fillna(0),
                        sub['pr_auc']+sub['pr_auc_std'].fillna(0), alpha=0.15, color=color)
        rand_val = RANDOM_AUPRC.get(p, np.nan)
        if not np.isnan(rand_val):
            ax.axhline(rand_val, color=color, linestyle='--', linewidth=1.0, alpha=0.7)
            rand_handles.append(Line2D([0], [0], color=color, linestyle='--', linewidth=1.2,
                                       label=f'Random AUPRC = {rand_val:.3f} ({p.replace("_"," ").title()})'))
    ax.set_xscale('log')
    ax.set_xlabel('Number of HVGs (log scale)')
    ax.set_ylabel('AUPRC')
    ax.set_title('HVG Sensitivity — AUPRC vs nHVGs')
    solid_handles, solid_labels = ax.get_legend_handles_labels()
    ax.legend(handles=solid_handles + rand_handles, fontsize=8, ncol=2)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'ICI_hvg_sensitivity.png', bbox_inches='tight', dpi=150)
    plt.show()

---
## COMPASS Fine-tuning Results (LFT / PFT / FFT)

In [15]:
COMPASS_FT_OUT   = MODEL_OUT / 'compass'
COMPASS_FT_MODES = ['LFT', 'PFT', 'FFT']


def load_compass_ft(paradigm, cohort_filter=None):
    base = COMPASS_FT_OUT / paradigm
    if not base.exists(): return pd.DataFrame()
    dfs = []
    for tag_dir in sorted(base.iterdir()):
        if not tag_dir.is_dir(): continue
        if cohort_filter and tag_dir.name not in cohort_filter: continue
        csv = tag_dir / 'all_metrics.csv'
        if csv.exists(): dfs.append(pd.read_csv(csv))
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


def aggregate_compass_ft(paradigm, cohort_filter=None):
    df = load_compass_ft(paradigm, cohort_filter)
    if df.empty or 'overall_pr_auc' not in df.columns: return pd.DataFrame()
    df = df[df['error'].fillna('') == '']
    if 'val_pr_auc' in df.columns:
        val_mean = (
            df.dropna(subset=['val_pr_auc'])
            .groupby(['cohort_tag','mode'])['val_pr_auc'].mean().reset_index()
        )
        best_mode = (
            val_mean.sort_values('val_pr_auc', ascending=False)
            .groupby('cohort_tag', as_index=False).head(1)[['cohort_tag','mode']]
        )
        df = df.merge(best_mode, on=['cohort_tag','mode'])
    return (
        df.dropna(subset=['overall_pr_auc'])
        .groupby(['cohort_tag','mode'])['overall_pr_auc']
        .agg(pr_auc_mean='mean', pr_auc_std='std', n='count').reset_index()
    )


def aggregate_compass_ft_by_mode(paradigm, cohort_filter=None):
    df = load_compass_ft(paradigm, cohort_filter)
    if df.empty or 'overall_pr_auc' not in df.columns: return pd.DataFrame()
    df = df[df['error'].fillna('') == '']
    return (
        df.dropna(subset=['overall_pr_auc'])
        .groupby('mode')['overall_pr_auc']
        .agg(pr_auc_mean='mean', pr_auc_std='std', n='count').reset_index()
        .sort_values('mode')
    )


transfer_tags = [f'{s}_to_{t}' for s in TRANSFER_TRAIN_COHORTS for t in ALL_COHORTS if t != s]

cft = {
    'within_cohort': aggregate_compass_ft('within_cohort', WITHIN_COHORTS),
    'loco':          aggregate_compass_ft('loco'),
    'transfer':      aggregate_compass_ft('transfer', transfer_tags),
}
for p, df in cft.items():
    print(f"{p}: {len(df)} rows, modes={df['mode'].unique().tolist()}" if not df.empty else f"{p}: no data")

cft_by_mode = {
    'within_cohort': aggregate_compass_ft_by_mode('within_cohort', WITHIN_COHORTS),
    'loco':          aggregate_compass_ft_by_mode('loco'),
    'transfer':      aggregate_compass_ft_by_mode('transfer', transfer_tags),
}
for p, df in cft_by_mode.items():
    if not df.empty:
        print(f"\n{p} (all modes):")
        print(df[['mode','pr_auc_mean','pr_auc_std','n']].to_string(index=False))


within_cohort: 2 rows, modes=['LFT']
loco: 5 rows, modes=['LFT', 'PFT']
transfer: 8 rows, modes=['LFT', 'PFT']

within_cohort (all modes):
mode  pr_auc_mean  pr_auc_std  n
 FFT     0.353330    0.094569 10
 LFT     0.361556    0.093686 10
 PFT     0.354592    0.092120 10

loco (all modes):
mode  pr_auc_mean  pr_auc_std  n
 FFT     0.345704    0.151480 25
 LFT     0.412126    0.153300 25
 PFT     0.408893    0.124371 25

transfer (all modes):
mode  pr_auc_mean  pr_auc_std  n
 FFT     0.353928    0.147177 40
 LFT     0.369293    0.148355 40
 PFT     0.367520    0.144206 40


In [16]:
# COMPASS frozen vs FT modes bar chart
MODE_PALETTE = {'LFT':'#BBDEFB','PFT':'#1976D2','FFT':'#0D47A1'}
FROZEN_COLOR = PALETTE['COMPASS']


def plot_compass_ft_bar(paradigm, ax=None):
    df_agg_p = agg.get(paradigm, pd.DataFrame())
    frozen_mean, frozen_std = np.nan, np.nan
    if not df_agg_p.empty:
        cr = df_agg_p[df_agg_p['RNA_key']=='COMPASS']
        if not cr.empty:
            best_combo = cr.groupby(['nHVGs', 'arch'])['overall_pr_auc_mean'].mean().idxmax()
            mask = (cr['nHVGs'] == best_combo[0]) & (cr['arch'] == best_combo[1])
            frozen_mean = cr.loc[mask, 'overall_pr_auc_mean'].mean()
            frozen_std  = cr.loc[mask, 'overall_pr_auc_std'].mean()
    mode_stats = {}
    df_mode = cft_by_mode.get(paradigm, pd.DataFrame())
    if not df_mode.empty:
        mode_stats = df_mode.set_index('mode').reindex(COMPASS_FT_MODES).dropna().to_dict('index')
    labels = ['Frozen'] + list(mode_stats.keys())
    means  = [frozen_mean] + [mode_stats[m]['pr_auc_mean'] for m in mode_stats]
    colors = [FROZEN_COLOR]+ [MODE_PALETTE[m] for m in mode_stats]
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 4))
    x = np.arange(len(labels))
    ax.bar(x, means, color=colors, edgecolor='white', linewidth=0.5)

    rand_val = RANDOM_AUPRC.get(paradigm, np.nan)
    if not np.isnan(rand_val):
        ax.axhline(rand_val, color='gray', linestyle='--', linewidth=1.2, zorder=5)
        rand_handle = Line2D([0], [0], color='gray', linestyle='--', linewidth=1.2,
                             label=f'Random AUPRC = {rand_val:.3f}')
        ax.legend(handles=[rand_handle], loc='lower right', fontsize=8, framealpha=0.8)

    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=25, ha='right', fontsize=9)
    ax.set_ylabel('AUPRC' if ax is axes[0] else '')
    ax.set_ylim(0.2, 0.55)
    #ax.set_title(paradigm.replace('_',' ').title())
    ax.grid(axis='y', alpha=0.3)
    for xi, m in enumerate(means):
        if not np.isnan(m):
            ax.text(xi, m + 0.01, f'{m:.3f}', ha='center', va='bottom', fontsize=8)


fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)
for ax, p in zip(axes, ['within_cohort','loco','transfer']):
    plot_compass_ft_bar(p, ax=ax)
#plt.suptitle('COMPASS: Frozen vs Fine-tuning Modes (mean AUPRC)', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'ICI_compass_ft_bar.png', bbox_inches='tight', dpi=150)
plt.show()

In [17]:
# COMPASS FT summary table
rows = []
for p in ['within_cohort', 'loco', 'transfer']:
    df_agg_p = agg.get(p, pd.DataFrame())
    if not df_agg_p.empty:
        cr = df_agg_p[df_agg_p['RNA_key']=='COMPASS']
        if not cr.empty:
            best_combo = cr.groupby(['nHVGs', 'arch'])['overall_pr_auc_mean'].mean().idxmax()
            mask = (cr['nHVGs'] == best_combo[0]) & (cr['arch'] == best_combo[1])
            rows.append({'paradigm':p,'model':'COMPASS (frozen)',
                         'pr_auc_mean':cr.loc[mask,'overall_pr_auc_mean'].mean(),
                         'pr_auc_std': cr.loc[mask,'overall_pr_auc_std'].mean()})
    df_mode_p = cft_by_mode.get(p, pd.DataFrame())
    if not df_mode_p.empty:
        for _, mrow in df_mode_p.iterrows():
            rows.append({'paradigm':p,'model':f'COMPASS-{mrow["mode"]}',
                         'pr_auc_mean':mrow['pr_auc_mean'],'pr_auc_std':mrow['pr_auc_std']})

summary = pd.DataFrame(rows)
if not summary.empty:
    summary['value'] = summary.apply(
        lambda r: f"{r['pr_auc_mean']:.3f} +/- {r['pr_auc_std']:.3f}"
                  if pd.notna(r['pr_auc_std']) else f"{r['pr_auc_mean']:.3f}", axis=1)
    pivot = summary.pivot(index='model', columns='paradigm', values='value')
    pivot = pivot.reindex([m for m in ['COMPASS (frozen)','COMPASS-LFT','COMPASS-PFT','COMPASS-FFT'] if m in pivot.index])
    pivot.columns.name = None
    display(pivot)

,loco,transfer,within_cohort
model,,,
COMPASS (frozen),0.382 +/- 0.046,0.393 +/- 0.041,0.486 +/- 0.123
COMPASS-LFT,0.412 +/- 0.153,0.369 +/- 0.148,0.362 +/- 0.094
COMPASS-PFT,0.409 +/- 0.124,0.368 +/- 0.144,0.355 +/- 0.092
COMPASS-FFT,0.346 +/- 0.151,0.354 +/- 0.147,0.353 +/- 0.095


In [18]:
# 3-way comparison: frozen COMPASS vs best FT COMPASS vs best baseline (per paradigm)
BASELINE_CATS = {'GS', 'HVG', 'PCA', 'Raw_expression'}
rows = []
for p in ['within_cohort', 'loco', 'transfer']:
    df_agg_p = agg.get(p, pd.DataFrame())
    df_ft_p  = cft.get(p, pd.DataFrame())
    if df_agg_p.empty: continue
    cr = df_agg_p[df_agg_p['RNA_key'] == 'COMPASS']
    if not cr.empty:
        best_combo = cr.groupby(['nHVGs', 'arch'])['overall_pr_auc_mean'].mean().idxmax()
        mask = (cr['nHVGs'] == best_combo[0]) & (cr['arch'] == best_combo[1])
        frozen_mean = cr.loc[mask, 'overall_pr_auc_mean'].mean()
    else:
        frozen_mean = np.nan
    ft_mean = df_ft_p['pr_auc_mean'].mean() if not df_ft_p.empty else np.nan
    bl_df = df_agg_p[df_agg_p['category'].isin(BASELINE_CATS)]
    best_cat, bl_mean = '', np.nan
    if not bl_df.empty:
        combo_means = (bl_df.groupby(['category', 'RNA_key', 'nHVGs', 'arch'])['overall_pr_auc_mean']
                       .mean().reset_index()
                       .sort_values('overall_pr_auc_mean', ascending=False))
        best_row = combo_means.iloc[0]
        best_cat = best_row['category']
        bl_mean  = bl_df[
            (bl_df['RNA_key'] == best_row['RNA_key']) &
            (bl_df['nHVGs']   == best_row['nHVGs'])   &
            (bl_df['arch']    == best_row['arch'])
        ]['overall_pr_auc_mean'].mean()
    rows.append({'paradigm': p, 'frozen_mean': frozen_mean, 'ft_mean': ft_mean,
                 'bl_mean': bl_mean, 'best_cat': best_cat})

df_cmp = pd.DataFrame(rows)
if df_cmp.empty:
    print('No data yet')
else:
    plabels = {'within_cohort': 'Within Cohort', 'loco': 'LOCO', 'transfer': 'Transfer'}
    x, w = np.arange(len(df_cmp)), 0.25
    fig, ax = plt.subplots(figsize=(9, 5))
    b1 = ax.bar(x - w, df_cmp['frozen_mean'], w, color=PALETTE['COMPASS'],
                edgecolor='white', linewidth=0.5, label='COMPASS (frozen)')
    b2 = ax.bar(x,     df_cmp['ft_mean'],     w, color='#1976D2',
                edgecolor='white', linewidth=0.5, label='COMPASS (fine-tuned, best mode)')
    b3 = ax.bar(x + w, df_cmp['bl_mean'],     w, color='#FF9800',
                edgecolor='white', linewidth=0.5, label='Best baseline')
    for bars, col in [(b1, 'frozen_mean'), (b2, 'ft_mean')]:
        for bar, v in zip(bars, df_cmp[col]):
            if not np.isnan(v):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                        f'{v:.3f}', ha='center', va='bottom', fontsize=7)
    for i, row in df_cmp.iterrows():
        if not np.isnan(row['bl_mean']):
            ax.text(x[i] + w, row['bl_mean'] + 0.03, f'{row["bl_mean"]:.3f}',
                    ha='center', va='bottom', fontsize=7)
            ax.text(x[i] + w, row['bl_mean'] + 0.01, f'({row["best_cat"]})',
                    ha='center', va='bottom', fontsize=6, color='#444444', style='italic')

    # Per-paradigm random AUPRC: short horizontal segment spanning each group
    rand_handles = []
    for i, row in df_cmp.iterrows():
        rand_val = RANDOM_AUPRC.get(row['paradigm'], np.nan)
        if not np.isnan(rand_val):
            ax.hlines(rand_val, x[i] - w*1.6, x[i] + w*1.6,
                      colors='gray', linestyles='--', linewidth=1.5, zorder=5)
            rand_handles.append(Line2D([0], [0], color='gray', linestyle='--', linewidth=1.5,
                                       label=f'Random AUPRC = {rand_val:.3f} ({plabels[row["paradigm"]]})'))

    ax.set_xticks(x)
    ax.set_xticklabels([plabels[p] for p in df_cmp['paradigm']], fontsize=11)
    ax.set_ylabel('AUPRC')
    ax.set_ylim(0.2, 0.9)
    #ax.set_title('COMPASS Frozen vs Fine-tuned vs Best Baseline (per paradigm)')
    bar_handles, bar_labels = ax.get_legend_handles_labels()
    ax.legend(handles=bar_handles + rand_handles, fontsize=8, ncol=2)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'ICI_compass_comparison.png', bbox_inches='tight', dpi=150)
    plt.show()
    print("\nBest baseline per paradigm:")
    for _, row in df_cmp.iterrows():
        print(f"  {row['paradigm']}: {row['best_cat']} = {row['bl_mean']:.3f}  "
              f"(frozen COMPASS: {row['frozen_mean']:.3f}, FT COMPASS: {row['ft_mean']:.3f})")


Best baseline per paradigm:
  within_cohort: GS = 0.485  (frozen COMPASS: 0.486, FT COMPASS: 0.362)
  loco: GS = 0.369  (frozen COMPASS: 0.382, FT COMPASS: 0.418)
  transfer: GS = 0.392  (frozen COMPASS: 0.393, FT COMPASS: 0.372)
